# 🛰️ Aurora — Notebook 02: Multispectral Model Training

**Project:** Aurora — Geo Snap Paradigm  
**Objective:** Train and evaluate four model architectures on EuroSAT RGB and Multispectral data.

## Models
| # | Architecture | Modality | Strategy |
|---|---|---|---|
| 1 | **EfficientNet-B0** | RGB | Pre-trained ImageNet, fine-tuned |
| 2 | **Swin Transformer** | RGB | Pre-trained ImageNet + SWA + CosineAnneal |
| 3 | **ResNet-50 (Surgery)** | 13-band MS | Weight surgery + mean init |
| 4 | **BalancedAttentionNet** | 13-band MS | Training from scratch |

---
> ⚠️ **Data Required:** Place EuroSAT data under `../data/`. Run **Notebook 01** first to generate split indices.

In [ ]:
# ─── Setup ───────────────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from pathlib import Path
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

from dataset import EuroSATRGBDataset, EuroSATMSDataset, CutMixCollator, EUROSAT_CLASSES
from models import (
    get_efficientnet_b0,
    get_swin_transformer,
    construct_multispectral_resnet50,
    BalancedAttentionNet,
    SWAWrapper,
    LabelSmoothingCrossEntropy,
)
from utils import train_one_epoch, evaluate, plot_training_curves, per_class_accuracy

plt.style.use('dark_background')

# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else
                      'mps'  if torch.backends.mps.is_available() else 'cpu')
print(f'✅ Device: {DEVICE}')
print(f'✅ PyTorch version: {torch.__version__}')

In [ ]:
# ─── Configuration ────────────────────────────────────────────────────────────
CFG = dict(
    rgb_root     = Path('../Data/EuroSAT'),
    ms_root      = Path('../Data/EuroSATallBands'),
    output_dir   = Path('../outputs'),
    batch_size   = 64,
    num_workers  = 4,
    num_classes  = 10,
    label_smooth = 0.1,
    # Training params per model
    efficientnet = dict(lr=1e-3, epochs=30, weight_decay=1e-4),
    swin         = dict(lr=5e-5, epochs=40, weight_decay=1e-4, swa_start=25),
    resnet50_ms  = dict(lr=5e-4, epochs=35, weight_decay=1e-4),
    balanced_attn= dict(lr=1e-3, epochs=50, weight_decay=5e-4),
)
CFG['output_dir'].mkdir(exist_ok=True)

# Load spatial split indices from Notebook 01 (or use full dataset)
if Path('../outputs/train_indices.npy').exists():
    TRAIN_IDX = np.load('../outputs/train_indices.npy').tolist()
    VAL_IDX   = np.load('../outputs/val_indices.npy').tolist()
    print(f'📂 Loaded spatial splits → Train: {len(TRAIN_IDX):,} | Val: {len(VAL_IDX):,}')
else:
    TRAIN_IDX, VAL_IDX = None, None
    print('⚠️  No split indices found. Using full dataset (run Notebook 01 first).')

DATA_AVAILABLE = CFG['rgb_root'].exists() or CFG['ms_root'].exists()
print(f'Data available: {DATA_AVAILABLE}')

In [ ]:
# ─── Load Checkpoints & Build Val DataLoader ──────────────────────────────────
import numpy as np, torch
from pathlib import Path
from PIL import Image
from torchvision import transforms
from torch.utils.data import DataLoader
from dataset import CLASS_TO_IDX, EUROSAT_CLASSES
from models import get_swin_transformer, construct_multispectral_resnet50
import rasterio

DEVICE = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
MEAN = np.load('../outputs/rgb_means.npy').tolist()
STD  = np.load('../outputs/rgb_stds.npy').tolist()

# ── Rebuild RGB val set (same seed as train_swin.py) ─────────────────────────
rgb_files, rgb_labels = [], []
for root in ['../Data/EuroSAT/train', '../Data/EuroSAT/val']:
    for cls_dir in sorted(Path(root).iterdir()):
        if not cls_dir.is_dir(): continue
        lbl = CLASS_TO_IDX.get(cls_dir.name)
        if lbl is None: continue
        for f in sorted(cls_dir.iterdir()):
            if f.suffix.lower() in ('.jpg', '.jpeg', '.png'):
                rgb_files.append(f); rgb_labels.append(lbl)

np.random.seed(42)
idx = np.random.permutation(len(rgb_files))
val_idx = idx[:int(0.2*len(rgb_files))]

val_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class SimpleRGB(torch.utils.data.Dataset):
    def __init__(self, files, labels, tf):
        self.files, self.labels, self.tf = files, labels, tf
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        return self.tf(Image.open(self.files[i]).convert('RGB')), self.labels[i]

val_dl = DataLoader(SimpleRGB([rgb_files[i] for i in val_idx],
                               [rgb_labels[i] for i in val_idx], val_tf),
                    batch_size=64, shuffle=False, num_workers=0)
print(f'RGB val samples: {len(val_idx):,}')

# ── Load Swin-T ───────────────────────────────────────────────────────────────
swin = get_swin_transformer(num_classes=10, pretrained=False).to(DEVICE)
swin.load_state_dict(torch.load('../outputs/swin_transformer_rgb_best.pth', map_location=DEVICE))
swin.eval()
print('Loaded: swin_transformer_rgb_best.pth')

all_preds, all_true = [], []
with torch.no_grad():
    for imgs, lbls in val_dl:
        all_preds.extend(swin(imgs.to(DEVICE)).argmax(1).cpu().numpy())
        all_true.extend(lbls.numpy())
rgb_acc = (np.array(all_preds)==np.array(all_true)).mean()
print(f'Swin-T RGB Val Accuracy: {rgb_acc*100:.2f}%')


In [ ]:
# ─── Confusion Matrix — Swin-T RGB ───────────────────────────────────────────
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(all_true, all_preds)
fig, ax = plt.subplots(figsize=(12, 10), facecolor='#0d1117')
ax.set_facecolor('#161b22')
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=EUROSAT_CLASSES)
disp.plot(ax=ax, colorbar=True, cmap='Blues', xticks_rotation=45)
ax.set_title('Confusion Matrix — Swin-T RGB (EuroSAT Val Set)',
             color='white', fontsize=13, fontweight='bold')
for txt in ax.texts: txt.set_color('white')
ax.xaxis.label.set_color('white'); ax.yaxis.label.set_color('white')
ax.tick_params(colors='white')
plt.tight_layout()
plt.savefig('../outputs/xai/confusion_matrix.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Saved → ../outputs/xai/confusion_matrix.png')


In [ ]:
# ─── Load ResNet-50 MS Checkpoint & Evaluate ─────────────────────────────────
MS_MEANS = np.load('../outputs/ms_means.npy')
MS_STDS  = np.load('../outputs/ms_stds.npy')

ms_files, ms_labels = [], []
for root in ['../Data/EuroSATallBands/train', '../Data/EuroSATallBands/val']:
    for cls_dir in sorted(Path(root).iterdir()):
        if not cls_dir.is_dir(): continue
        lbl = CLASS_TO_IDX.get(cls_dir.name)
        if lbl is None: continue
        for f in sorted(cls_dir.glob('*.tif')):
            ms_files.append(f); ms_labels.append(lbl)

np.random.seed(42)
ms_idx = np.random.permutation(len(ms_files))
ms_val_idx = ms_idx[:int(0.2*len(ms_files))]

class SimpleMS(torch.utils.data.Dataset):
    def __init__(self, files, labels):
        self.files, self.labels = files, labels
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        with rasterio.open(self.files[i]) as src:
            raw = src.read().astype(np.float32)
        normed = (raw - MS_MEANS[:,None,None]) / (MS_STDS[:,None,None] + 1e-8)
        return torch.from_numpy(normed), self.labels[i]

ms_val_dl = DataLoader(SimpleMS([ms_files[i] for i in ms_val_idx],
                                 [ms_labels[i] for i in ms_val_idx]),
                       batch_size=32, shuffle=False, num_workers=0)
print(f'MS val samples: {len(ms_val_idx):,}')

ms_model = construct_multispectral_resnet50(num_classes=10, in_channels=13).to(DEVICE)
ms_model.load_state_dict(torch.load('../outputs/resnet50_ms_surgery_best.pth', map_location=DEVICE))
ms_model.eval()
print('Loaded: resnet50_ms_surgery_best.pth')

ms_preds, ms_true = [], []
with torch.no_grad():
    for imgs, lbls in ms_val_dl:
        ms_preds.extend(ms_model(imgs.to(DEVICE)).argmax(1).cpu().numpy())
        ms_true.extend(lbls.numpy())
ms_acc = (np.array(ms_preds)==np.array(ms_true)).mean()
print(f'ResNet-50 MS Val Accuracy: {ms_acc*100:.2f}%')


In [ ]:
# ─── Results Comparison Table ─────────────────────────────────────────────────
import pandas as pd, matplotlib.pyplot as plt

results = pd.DataFrame([
    {'Model': 'Swin Transformer', 'Modality': 'RGB (3-band)',  'Strategy': 'Pre-trained + SWA', 'Val Acc (%)': round(rgb_acc*100,2), 'Params (M)': 28.3},
    {'Model': 'ResNet-50',        'Modality': 'MS (13-band)',  'Strategy': 'Weight Surgery',    'Val Acc (%)': round(ms_acc*100,2),  'Params (M)': 25.6},
])
print(results.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4), facecolor='#0d1117')
ax.set_facecolor('#161b22')
bars = ax.barh(results['Model'] + ' (' + results['Modality'] + ')',
               results['Val Acc (%)'], color=['#58a6ff','#3fb950'], alpha=0.85, height=0.4)
ax.set_xlabel('Validation Accuracy (%)', color='#8b949e')
ax.set_title('Aurora — Model Accuracy Comparison', color='white', fontsize=13, fontweight='bold')
ax.set_xlim(96, 101)
ax.grid(True, axis='x', alpha=0.3)
for bar, row in zip(bars, results.itertuples()):
    ax.text(row[4]+0.02, bar.get_y()+bar.get_height()/2.,
            f'{row[4]:.2f}%', va='center', color='white', fontsize=10)
plt.tight_layout()
plt.savefig('../outputs/02_model_comparison.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Saved → ../outputs/02_model_comparison.png')


In [ ]:
# ─── Generate Prediction CSVs ─────────────────────────────────────────────────
from torch.utils.data import DataLoader
from dataset import EuroSATTestDataset, EUROSAT_CLASSES
from utils import generate_test_predictions
 
RGB_TEST_DIR = Path('../Data/EuroSAT_test_flat')
MS_TEST_DIR  = Path('../Data/EuroSATallBands_test_flat')
 
# ── RGB predictions ──────────────────────────────────────────────────────────
if RGB_TEST_DIR.exists():
    print(f'▶ Generating RGB predictions from {RGB_TEST_DIR}...')
 
    # Load best RGB model (Swin if available, else EfficientNet)
    swin_ckpt = Path('../outputs/swin_transformer_rgb_swa.pth')
    effnet_ckpt = Path('../outputs/efficientnet_b0_rgb_best.pth')
 
    best_rgb = get_swin_transformer(num_classes=10, pretrained=False) if (swin_ckpt.exists() and TIMM_OK) \
               else get_efficientnet_b0(num_classes=10, pretrained=False)
    ckpt = swin_ckpt if swin_ckpt.exists() else effnet_ckpt
    if ckpt.exists():
        best_rgb.load_state_dict(torch.load(ckpt, map_location='cpu'))
        print(f'   Loaded checkpoint: {ckpt}')
    else:
        print('   ⚠️  No checkpoint found. Predictions will be random (train first).')
 
    rgb_test_ds = EuroSATTestDataset(RGB_TEST_DIR, is_ms=False)
    rgb_test_dl = DataLoader(rgb_test_ds, batch_size=64, shuffle=False, num_workers=2)
 
    rgb_pred_df = generate_test_predictions(
        best_rgb, rgb_test_dl, DEVICE,
        '../outputs/rgb_predictions.csv',
        EUROSAT_CLASSES,
    )
    print(rgb_pred_df.head())
else:
    print(f'⚠️  RGB test dir not found: {RGB_TEST_DIR}')
    print('   Place EuroSAT_test_flat/ under data/ and re-run.')
 
# ── MS predictions ───────────────────────────────────────────────────────────
if MS_TEST_DIR.exists():
    print(f'\n▶ Generating MS predictions from {MS_TEST_DIR}...')
 
    ms_ckpt = Path('../outputs/resnet50_ms_surgery_best.pth')
    best_ms = construct_multispectral_resnet50(num_classes=10)
    if ms_ckpt.exists():
        best_ms.load_state_dict(torch.load(ms_ckpt, map_location='cpu'))
        print(f'   Loaded checkpoint: {ms_ckpt}')
    else:
        print('   ⚠️  No MS checkpoint found. Train first.')
 
    ms_test_ds = EuroSATTestDataset(MS_TEST_DIR, is_ms=True)
    ms_test_dl = DataLoader(ms_test_ds, batch_size=32, shuffle=False, num_workers=2)
 
    ms_pred_df = generate_test_predictions(
        best_ms, ms_test_dl, DEVICE,
        '../outputs/ms_predictions.csv',
        EUROSAT_CLASSES,
    )
    print(ms_pred_df.head())
else:
    print(f'⚠️  MS test dir not found: {MS_TEST_DIR}')
    print('   Place EuroSATallBands_test_flat/ under data/ and re-run.')